# RealSaS — Mage FIT2 Geppetto Corrected-Substrate Refit — Public Run-All V1

**Authority:** `MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1`  
**Mode:** fresh Geppetto fit from corrected Mage substrate; historical Geppetto checkpoint is never loaded.

Run this notebook with **Runtime → Run all** on a Colab **A100** runtime. The notebook:

- mounts Drive,
- clones the now-public `merynz/RealSaS-OPT` branch without a token,
- verifies the exact corrected H1/GSA8192 authority and all pinned hashes,
- performs a fresh Geppetto refit with the frozen scientific thresholds,
- refuses to authorize Arachne unless Geppetto reaches 48/48 terminal stability,
- writes and displays the required **real V0..V7 qualified-skeleton evidence**,
- emits a SHA manifest, evidence ZIP, and fail-closed Arachne handoff.

No generated/synthetic image is accepted as evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, json, hashlib, subprocess, shutil, time, zipfile
from pathlib import Path

# GPU gate: this refit is intended for the Colab A100 path.
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True))
import torch
assert torch.cuda.is_available(), 'CUDA_REQUIRED_FOR_GEPPETTO_FIT2_REFIT'
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print('torch=', torch.__version__, 'gpu=', gpu_name, 'memory_GiB=', round(gpu_gib, 1))
assert gpu_gib >= 30.0, f'GPU_MEMORY_TOO_SMALL_FOR_PINNED_FIT2_REFIT::{gpu_gib:.1f}GiB; use Colab A100'


In [ ]:
# Public repo: clean clone, no PAT, no interactive GitHub prompt.
REPO = Path('/content/RealSaS-OPT')
BRANCH = 'repair/mage-full-subject-reclosure-20260912'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--depth','1','--branch',BRANCH,
    'https://github.com/merynz/RealSaS-OPT.git', str(REPO)
], check=True)
HEAD = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('repo_head=', HEAD)

authority_path = REPO / 'canonical/MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1.json'
runner_path = REPO / 'experiments/mage_full_subject_reclosure_v1/run_geppetto_fit2_corrected_substrate_refit_v1.py'
assert authority_path.is_file(), 'FIT2 authority missing from cloned branch'
assert runner_path.is_file(), 'FIT2 Geppetto runner missing from cloned branch'
authority = json.loads(authority_path.read_text())
assert authority['status'] == 'ACTIVE__FIT2_PIPELINE_REFIT_AUTHORITY'
assert authority['authority_id'] == 'MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1'
print(json.dumps({
    'authority_id': authority['authority_id'],
    'gsa_lineage': authority['active_upstream_authority']['gsa_lineage_hash'],
    'gsa_nodes': authority['active_upstream_authority']['gsa_actual_nodes'],
    'gsa_relations': authority['active_upstream_authority']['gsa_relations'],
    'geppetto_mode': authority['geppetto_refit']['mode'],
}, indent=2))


In [ ]:
# Runtime dependencies only. Torch is supplied by Colab.
subprocess.run([sys.executable,'-m','pip','install','-q','scipy','pillow','matplotlib'], check=True)

# Exact Drive paths: no whole-Drive crawling and no manual file selection.
MY = Path('/content/drive/MyDrive')
RECLOSURE = MY / 'REALSAS_MAGE_FULL_SUBJECT_RECLOSURE_20260912'
INPUT_ROOT = RECLOSURE / 'IRIS_H1_V2_INPUT' / '20260912_FULL_SUBJECT'
ZERO = RECLOSURE / 'IRIS_H1_V2_CONTINUATION_RUNS' / '20260912T074348Z' / 'ZERO_SURFACE_PRODUCT_CLIPPED.npz'
NORMALIZED = MY / 'RealSaS_MASTER_CORPUS_1024_V3' / 'master' / 'variants' / 'kaykit_cc0_cf898585da33fab50c724d31' / 'normalized.npz'
CAMERAS = [INPUT_ROOT / f'V{i}.camera.json' for i in range(8)]
OBS = [INPUT_ROOT / f'V{i}.png' for i in range(8)]

ZERO_SHA='56073e8b348b828350c812ac44982b823237196d5ec2f361241877e9ae301925'
NORM_SHA='528bef491eceb358ebc8ecb2a46af1d37b4322a7ef500281403a8207fe7c648f'
CAM_SHA=[
'73004e0654b576e0c51893af544e0af8fcc4e613ce07ea9884272285d55cd541',
'bdc172a4aff332f956d1403e36b2f8684b68059fdc82f9efddf35d05a6d9b4d4',
'3c2bbc44ef9005b4a545a3381205a5d6a92af15b4791c9075071b8cad02a1a6c',
'24b2f115d908422d885f85e956fcc36ac78fd0c90b503f698caa62febc2b9c4d',
'5bf00783d6509c2ca142e05ef705d5cdb5df17ad248b782d2fe8cf8a297bee39',
'7ee3e50739318eeb122b5b0ec67260dd32e21d949398f48c408a6c239e5c89fe',
'daa19fa58ff602977d64b720c4198956809855149d814df487c7762a963f1eec',
'68f51fbfce4c31f94281e1569d74b44609435285668f8a8b1b278e76db6ea53f']
OBS_SHA=[
'8e9875c16bba3047c8f2fc211b984f2399d1145f5720c7427c6fc03a3fec0616',
'fa94283780d5e5f3ad3943bbffc1f0592a70fc362fdd03b94a1d1cb46889dc30',
'777bf4f7c505b405d3a5d2a111b2f297949454f77cae7c33064bf02479d384a5',
'2a771c91c1d1c0b75dab14f6e98b7905a8429bbf0c0a8dd690261f93f6476d86',
'354bb239feb6c1a515917fee4efb42fb1e0cb9f173d0eb3191fb0901a02a6228',
'158fc14a75aa69f6133f2ba3ec5df2ad146afc62f477d7d3b4a41ca2cb0e42f2',
'e3b08836c187863819d8b8aaa53aef79eddfee167e1fabf3fb1dd8ec0484563a',
'87d4ad46edffec3c6bff034d305194820288d3cc6ef9a9480ac2234fb56451bc']

def sha256(path: Path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(8<<20), b''):
            h.update(b)
    return h.hexdigest()

def require(path, expected, label):
    assert path.is_file(), f'{label}_MISSING::{path}'
    got=sha256(path)
    assert got==expected, f'{label}_SHA_DRIFT::{got}::{expected}'

require(ZERO, ZERO_SHA, 'ZERO_SURFACE')
require(NORMALIZED, NORM_SHA, 'NORMALIZED_CORPUS')
for i,p in enumerate(CAMERAS): require(p, CAM_SHA[i], f'CAMERA_V{i}')
for i,p in enumerate(OBS): require(p, OBS_SHA[i], f'OBSERVATION_V{i}')
print('PINNED_INPUTS_PASS')
print('ZERO=', ZERO)
print('NORMALIZED=', NORMALIZED)
print('INPUT_ROOT=', INPUT_ROOT)


In [ ]:
# Timestamped scientific output directory under the FIT2 Drive authority.
stamp=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
OUT_ROOT = RECLOSURE / 'MAGE_FIT2_PIPELINE_REFIT' / 'GEPPETTO_REFIT'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT / stamp
OUT.mkdir(parents=True, exist_ok=False)

RUNNER = runner_path
base=[
    sys.executable, str(RUNNER),
    '--zero-surface', str(ZERO),
    '--normalized-corpus', str(NORMALIZED),
    '--cameras', *[str(p) for p in CAMERAS],
    '--observations', *[str(p) for p in OBS],
    '--output-dir', str(OUT),
]

run_identity={
    'schema':'RealSaS.MageFIT2.GeppettoRunIdentity.v1',
    'repo_head':HEAD,
    'authority_id':'MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1',
    'output_dir':str(OUT),
    'training_mode':'FRESH_FROM_SCRATCH__NO_HISTORICAL_GEPPETTO_CHECKPOINT',
}
(OUT/'RUN_IDENTITY.json').write_text(json.dumps(run_identity,indent=2,sort_keys=True)+'\n')
print('OUT=',OUT)


In [ ]:
# Fail-closed preflight before any optimizer step.
subprocess.run(base+['--preflight-only'], cwd=REPO, check=True)
preflight_path=OUT/'GEPPETTO_FIT2_CORRECTED_SUBSTRATE_PREFLIGHT.json'
preflight=json.loads(preflight_path.read_text())
assert preflight['status']=='PASS__FIT2_CORRECTED_SUBSTRATE__FRESH_GEPPETTO_REFIT_READY'
assert preflight['historical_checkpoint_loaded'] is False
assert preflight['thresholds_changed'] is False
assert preflight['gsa_lineage_hash']=='65319061d802c640717010dddf0fd71a66ee6bd2fd31f6e614386f4d2584d5da'
assert preflight['surface_node_count']==8171
assert preflight['surface_edge_count']==23656
assert preflight['tensorization_hash']=='fe351362e195164805cebb0f63b74d1861ef123458b20ac56607016e00a6c67e'
assert preflight['observed_nodes']==7391
assert preflight['completed_nodes']==780
assert preflight['observation_support_counts']==[2537,2765,2194,3021,2772,2871,2183,2803]
assert preflight['target_count']==22
print(json.dumps(preflight, indent=2))


In [ ]:
# Fresh Geppetto refit. Output is preserved verbatim; console shows compact scientific progress.
log_path=OUT/'GEPPETTO_FIT2_TRAINING_STDOUT.jsonl'
with log_path.open('w', encoding='utf-8') as log:
    proc=subprocess.Popen(base, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line); log.flush()
        s=line.strip()
        if s.startswith('GEPPETTO_FIT2_CHECK='):
            row=json.loads(s.split('=',1)[1])
            loss=row.get('losses',{}).get('total')
            seeds=row.get('diffusion_seed_reports',[])
            seed_pass=sum(bool(x.get('pass')) for x in seeds)
            print(f"step={row['step']:5d} seeds={seed_pass}/4 pass={row['pass']} streak={row.get('terminal_streak',0):2d}/48 loss={loss}", flush=True)
        elif s.startswith('FIT2_') or s.startswith('PASS__'):
            print(s, flush=True)
    rc=proc.wait()
if rc!=0:
    tail='\n'.join(log_path.read_text(errors='replace').splitlines()[-80:])
    print(tail)
    raise RuntimeError(f'GEPPETTO_FIT2_TRAINING_FAILED_RC_{rc}::see::{log_path}')
print('GEPPETTO_FIT2_TRAINING_PROCESS_COMPLETE')


In [ ]:
# Terminal scientific gate + mandatory real eight-view evidence.
from IPython.display import display
from PIL import Image

result_path=OUT/'GEPPETTO_FIT2_CORRECTED_SUBSTRATE_RESULT.json'
assert result_path.is_file(), 'RESULT_JSON_MISSING'
result=json.loads(result_path.read_text())
assert result['status']=='FIT2_GEPPETTO_TERMINAL_PASS', result['status']
assert result['historical_checkpoint_loaded'] is False
assert result['thresholds_changed'] is False
assert result['terminal_streak'] >= 48
assert result['arachne_refit_authorized'] is True

contact=OUT/'GEPPETTO_FIT2_8VIEW_SKELETON_CONTACT_SHEET.png'
expected_views=[OUT/f'GEPPETTO_FIT2_V{i}_SKELETON_OVERLAY.png' for i in range(8)]
assert all(p.is_file() for p in expected_views), 'V0..V7_REAL_SKELETON_EVIDENCE_MISSING'
assert contact.is_file(), '8VIEW_CONTACT_SHEET_MISSING'

print('closure_step=', result['closure_step'])
print('checkpoint_sha256=', result['checkpoint_sha256'])
print('terminal_streak=', result['terminal_streak'])
print('eight_view_files=', [p.name for p in expected_views])
display(Image.open(contact))


In [ ]:
# Seal evidence, zip the run, and emit the only permitted Arachne handoff.
handoff={
  'schema':'RealSaS.MageFIT2.ArachneInputHandoff.v1',
  'status':'AUTHORIZED_BY_GEPPETTO_FIT2_TERMINAL_PASS',
  'repo_head':HEAD,
  'authority_id':'MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1',
  'gsa_lineage_hash':result['gsa_lineage_hash'],
  'tensorization_hash':result['tensorization_hash'],
  'geppetto_checkpoint_path':str(OUT/'GEPPETTO_FIT2_CORRECTED_SUBSTRATE_CHECKPOINT.pt'),
  'geppetto_checkpoint_sha256':result['checkpoint_sha256'],
  'geppetto_result_path':str(result_path),
  'eight_view_skeleton_contact_sheet':str(contact),
  'arachne_refit_authorized':True,
}
handoff_path=OUT/'ARACHNE_FIT2_INPUT_HANDOFF.json'
handoff_path.write_text(json.dumps(handoff,indent=2,sort_keys=True)+'\n')

# SHA-256 manifest over all current output artifacts.
manifest={}
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name not in {'GEPPETTO_FIT2_EVIDENCE_SHA256.json','GEPPETTO_FIT2_EVIDENCE_BUNDLE.zip'}:
        manifest[p.name]={'sha256':sha256(p),'bytes':p.stat().st_size}
manifest_path=OUT/'GEPPETTO_FIT2_EVIDENCE_SHA256.json'
manifest_path.write_text(json.dumps({
    'schema':'RealSaS.MageFIT2.GeppettoEvidenceManifest.v1',
    'repo_head':HEAD,
    'authority_id':'MAGE_FIT2_PIPELINE_REFIT_AUTHORITY_V1',
    'files':manifest,
},indent=2,sort_keys=True)+'\n')

zip_path=OUT/'GEPPETTO_FIT2_EVIDENCE_BUNDLE.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p != zip_path:
            z.write(p, arcname=p.name)

print(json.dumps(handoff, indent=2))
print('manifest=', manifest_path)
print('evidence_zip=', zip_path)
print('FIT2_GEPPETTO_STAGE_CLOSED_WITH_REAL_8VIEW_EVIDENCE')
